In [1]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np

from src.config import ExperimentConfig
from src.data_utils import load_market_data, summarize_market_data, get_field_panel
from src.features import build_features_from_config
from src.preprocessing import preprocess_features
from src.models import predict_stacked, stacked_predictions_to_panel
from src.persistence import load_final_artifacts
from src.evaluation import get_hourly_risk_free_rate, analyze_expected_returns

In [2]:
EXPERIMENT_NAME = "baseline_elastic_net_walk_forward"
ARTIFACTS_DIR = "artifacts"

In [3]:
artifacts = load_final_artifacts(
    experiment_name=EXPERIMENT_NAME,
    artifacts_dir=ARTIFACTS_DIR,
)

model = artifacts["model"]
config_dict = artifacts["config_dict"]
feature_columns = artifacts["feature_columns"]
metadata = artifacts["metadata"]

metadata

{'experiment_name': 'baseline_elastic_net_walk_forward',
 'model_name': 'elastic_net',
 'feature_set_name': 'baseline',
 'n_features': 36,
 'metrics': {'one_shot_sharpe_lag0': -1.6,
  'one_shot_turnover': 13.18,
  'walk_forward_sharpe_lag0': -0.43,
  'walk_forward_turnover': 12.15}}

In [4]:
config = ExperimentConfig.from_json(artifacts["config_path"])
config

ExperimentConfig(experiment_name='baseline_elastic_net_walk_forward', random_seed=0, paths=PathsConfig(data_dir='data/all/', artifacts_dir='artifacts', results_dir='results', in_sample_filename='data_in_sample.csv', test_filename='data_test.csv'), data=DataConfig(datetime_col=None, index_col=0, header_rows=[0, 1]), dates=DateConfig(start_date_train='2023-01-24', last_date_train='2024-01-24', start_date_validate='2024-01-25', last_date_validate='2024-07-24'), evaluation=EvalConfig(risk_free_rate_annual=0.05, transaction_cost=0.0, evaluation_lags=[0, 1, 2, 3, 6, 12], plot_option='matplotlib', annualization_factor=8760), features=FeatureConfig(feature_set_name='baseline', raw_fields=['return', 'close', 'nb_trades', 'volume_usd', 'funding_rate', 'open_interest_value'], feature_styles=['level', 'delta_1', 'shift_1', 'shift_6', 'mean_24', 'std_24'], use_engineered_features=False, engineered_feature_prefix='feature', max_nb_features=20, pct_engineered_features=0.25, fillna_value=0.0), preproc

In [5]:
test_data = load_market_data(
    filepath=f"{config.paths.data_dir}/{config.paths.test_filename}",
    index_col=config.data.index_col,
    header_rows=config.data.header_rows,
)

display(summarize_market_data(test_data))

FileNotFoundError: [Errno 2] No such file or directory: 'data/all//data_test.csv'

In [ ]:
test_features = build_features_from_config(test_data, config.features)

print("Raw test feature matrix shape:", test_features.shape)
display(test_features.head())

In [ ]:
test_features_processed = preprocess_features(test_features, config.preprocessing)

print("Processed test feature matrix shape:", test_features_processed.shape)
display(test_features_processed.head())

In [ ]:
missing_cols = [col for col in feature_columns if col not in test_features_processed.columns]
extra_cols = [col for col in test_features_processed.columns if col not in feature_columns]

print("Missing columns:", len(missing_cols))
print("Extra columns:", len(extra_cols))

if missing_cols:
    for col in missing_cols:
        test_features_processed[col] = 0.0

X_test = test_features_processed.reindex(columns=feature_columns)

print("Final X_test shape:", X_test.shape)

In [ ]:
test_predictions_stacked = predict_stacked(model, X_test)
test_predictions_panel = stacked_predictions_to_panel(test_predictions_stacked)

print("Stacked test predictions shape:", test_predictions_stacked.shape)
print("Panel test predictions shape:", test_predictions_panel.shape)

display(test_predictions_stacked.head())

In [ ]:
test_returns_panel = get_field_panel(test_data, "return")
rfr_hourly = get_hourly_risk_free_rate(config.evaluation.risk_free_rate_annual)

test_stats = analyze_expected_returns(
    expected_returns=test_predictions_panel,
    returns=test_returns_panel,
    rfr_hourly=rfr_hourly,
    title=f"{EXPERIMENT_NAME} - Test set",
    lags=[0],
    tc=0.0,
    plot_option=config.evaluation.plot_option,
    output_stats=True,
)

display(test_stats)

In [ ]:
official_test_sharpe_lag0 = test_stats.loc["Statistics", "sharpe"]
print("Official test Sharpe (lag 0, no transaction cost):", official_test_sharpe_lag0)